# Fake News Prediction System

## DawoodTech Capstone — Week 8 Final Project

This notebook documents the full machine learning lifecycle for binary fake news classification using the **ISOT Fake News Dataset**.

### Workflow
1. Data collection and cleaning
2. Exploratory data analysis
3. Feature engineering with TF-IDF
4. Model development (4 classifiers)
5. Model comparison and selection
6. Hyperparameter tuning with GridSearchCV
7. Model export for Streamlit deployment

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.evaluation import compute_metrics, cross_validate_model, plot_confusion_matrix, plot_model_comparison
from utils.models import MODEL_DISPLAY_NAMES, get_model_builders, optimize_model
from utils.preprocessing import TextPreprocessor, get_train_test_split
from train import generate_eda_plots

REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

sns.set_theme(style="whitegrid")
print(f"Project root: {PROJECT_ROOT}")

## 1. Dataset Preparation

Download `Fake.csv` and `True.csv` from Kaggle and place them in `data/raw/`.

Each file contains: `title`, `text`, `subject`, `date`.

In [ ]:
preprocessor = TextPreprocessor(data_dir=PROJECT_ROOT / "data" / "raw")
df = preprocessor.load_dataset()
dataset_info = preprocessor.dataset_summary(df)
dataset_info

In [ ]:
df.head()

## 2. Exploratory Data Analysis

We inspect class balance, text length distributions, correlations, and outliers.

In [ ]:
generate_eda_plots(df)

for image_name in [
    "target_distribution.png",
    "text_length_histogram.png",
    "word_count_boxplot.png",
    "correlation_heatmap.png",
    "title_vs_text_scatter.png",
]:
    display(plt.imread(FIGURES_DIR / image_name))
    plt.axis("off")
    plt.show()

### EDA Insights
- The dataset is relatively balanced between fake and real articles.
- Text length and word count vary widely; outliers above the 99th percentile are removed during preprocessing.
- Fake and real articles show different length patterns, which supports text-based classification.

## 3. Train/Test Split

In [ ]:
x_train, x_test, y_train, y_test = get_train_test_split(df)
len(x_train), len(x_test)

## 4. Model Development

We train four models required by the capstone rubric:
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Support Vector Machine

In [ ]:
comparison = {}
builders = get_model_builders()

for model_key, builder in builders.items():
    display_name = MODEL_DISPLAY_NAMES[model_key]
    model = builder()
    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)
    y_proba = model.predict_proba(x_test)[:, 1] if hasattr(model, "predict_proba") else None
    metrics = compute_metrics(y_test, y_pred, y_proba)
    cv_scores = cross_validate_model(model, x_train, y_train)

    comparison[display_name] = {
        "model_key": model_key,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "roc_auc": metrics.get("roc_auc"),
        **cv_scores,
    }

pd.DataFrame(comparison).T

In [ ]:
plot_model_comparison(
    {
        name: {
            "accuracy": values["accuracy"],
            "f1": values["f1"],
            "roc_auc": values.get("roc_auc", 0.0),
        }
        for name, values in comparison.items()
    },
    output_path=FIGURES_DIR / "model_comparison.png",
)
plt.imshow(plt.imread(FIGURES_DIR / "model_comparison.png"))
plt.axis("off")
plt.show()

## 5. Model Optimization

Select the best baseline model and tune it with `GridSearchCV`.

In [ ]:
best_name = max(
    comparison,
    key=lambda name: (comparison[name]["f1"], comparison[name].get("roc_auc", 0.0)),
)
best_key = comparison[best_name]["model_key"]
best_name, best_key

In [ ]:
search = optimize_model(best_key, x_train, y_train)
best_model = search.best_estimator_
search.best_params_, search.best_score_

In [ ]:
y_pred = best_model.predict(x_test)
y_proba = best_model.predict_proba(x_test)[:, 1]
optimized_metrics = compute_metrics(y_test, y_pred, y_proba)
optimized_metrics

## 6. Save Artifacts

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODELS_DIR / "best_model.joblib")

with (REPORTS_DIR / "model_comparison.json").open("w", encoding="utf-8") as file:
    json.dump(comparison, file, indent=2)

with (REPORTS_DIR / "optimized_metrics.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            **optimized_metrics,
            "best_params": search.best_params_,
            "best_cv_score": float(search.best_score_),
            "model_name": best_name,
            "model_key": best_key,
        },
        file,
        indent=2,
    )

print("Artifacts saved. Launch the app with: streamlit run app.py")